# FSCT 8561 — Lab 2
## Network Scanning & Enumeration — Build a Port Scanner in Python

**Course:** FSCT 8561 — Security Applications  
**Lab Duration:** 3 hours

### Overview
This lab builds directly on the raw TCP socket programming skills from Labs 0 and 1.

In Lab 1, you used a TCP client to connect to **one known server port**.

In Lab 2, you will turn that idea around:

> **Try a TCP connection → Observe the result → Move to the next port → Report what is exposed**

You will build a small TCP Connect port scanner using Python's built-in `socket` module. After you understand how the scanner works, you will compare your results with Nmap.

The focus is on:

- understanding what a port scanner actually does,
- reusing TCP socket concepts from previous labs,
- scanning a small, authorized port range,
- interpreting open and closed ports,
- adding timeouts and input validation,
- mapping common ports to likely services,
- comparing your Python scanner with a professional scanning tool,
- analyzing the security implications of exposed services.

> **Authorization requirement:** Scan only `127.0.0.1` / `localhost`, your own lab systems, or systems for which you have explicit permission.

## Learning Objectives

By the end of this lab, you should be able to:

- explain how TCP Connect port scanning works,
- use Python sockets to test whether a TCP port accepts a connection,
- scan a small range of ports using a loop,
- use socket return values and timeouts to interpret scan results,
- identify likely services associated with common ports,
- validate user-supplied targets and port ranges,
- compare a simple Python scanner with Nmap,
- analyze how exposed ports contribute to attack surface.

## Prerequisite

You must have completed **Lab 1**.

You should already understand:

- `socket.AF_INET`
- `socket.SOCK_STREAM`
- IP addresses and ports
- TCP connections
- loops
- `if / elif / else`
- `try / except`

Lab 2 deliberately reuses these concepts rather than introducing a large external scanning library immediately.

## Required Reading & Tutorials

- *Mastering Python for Networking and Security* — Chapter 8
- Nmap Reference Guide: `https://nmap.org/book/man.html`
- Python `socket` documentation: `https://docs.python.org/3/library/socket.html`

 In this Lab, you first build the core scanning logic yourself with raw Python sockets, then use Nmap for comparison.

# Part 1 — Connect Lab 1 to Port Scanning

In Lab 1, your client already did something very important:

```python
client_socket.connect((HOST, PORT))
```

You knew the server was listening on `PORT`, so the connection succeeded.

But what happens if you try to connect to a port where **no server is listening**?

That observation is the basis of a simple **TCP Connect scan**.

Conceptually:

```text
Choose a port
     ↓
Attempt TCP connection
     ↓
Did connection succeed?
   /             \
 Yes              No
  ↓                ↓
OPEN          Not open
```

A scanner repeats this process for several ports.

### Important distinction

A successful connection tells us that a TCP service is accepting connections on that port.

A failed connection does **not** automatically tell us everything about the reason. A port may be closed, traffic may be filtered, the host may be unreachable, or the connection may time out.

Our simple scanner will therefore be intentionally limited. Later, you will compare it with Nmap.

# Part 2 — New Python Concepts

Lab 2 introduces a few small Python ideas that we will use to build the scanner step by step.

### `socket.connect_ex()`

In Lab 1 you used:

```python
sock.connect((HOST, PORT))
```

For scanning, Python also provides:

```python
sock.connect_ex((HOST, PORT))
```

`connect_ex()` attempts a TCP connection but returns an error code instead of raising an exception for normal connection failures.

A return value of:

```text
0
```

means that the connection succeeded.

That makes it convenient for a basic TCP Connect scanner.

In [ ]:
import socket

target = "127.0.0.1"
port = 80

sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
sock.settimeout(0.5)

result = sock.connect_ex((target, port))

print("Return code:", result)

sock.close()

### `range()`

To scan several ports, we need to repeat the same operation.

```python
for port in range(20, 26):
    print(port)
```

Remember that the second number is not included, so this prints ports 20 through 25.

In [ ]:
for port in range(20, 26):
    print("Testing port:", port)

### `socket.settimeout()`

A scanner should not wait indefinitely for every connection attempt.

```python
sock.settimeout(0.5)
```

sets a short timeout for socket operations.

For this lab, use a reasonable timeout such as **0.5–1 second**. A very short timeout may produce misleading results on a slow network.

### `socket.gethostbyname()`

A user may enter either a hostname or an IPv4 address.

```python
socket.gethostbyname("localhost")
```

resolves the name to an IPv4 address.

We can place this inside `try / except` so that an invalid hostname does not crash the program.

In [ ]:
import socket

target = "localhost"

try:
    target_ip = socket.gethostbyname(target)
    print("Target IP:", target_ip)
except socket.gaierror:
    print("Unable to resolve target")

# Part 3 — Test ONE Port First

Do **not** begin by scanning hundreds of ports.

First, write a small program that tests **one TCP port**.

Create:

```text
scanner.py
```

Start with:

1. import `socket`,
2. define a target,
3. define one port,
4. create a TCP socket,
5. set a timeout,
6. call `connect_ex()`,
7. interpret the result,
8. close the socket.

Use `127.0.0.1` as your first target.

In [ ]:
import socket

TARGET = "127.0.0.1"
PORT = 80

scanner_socket = socket.socket(
    socket.AF_INET,
    socket.SOCK_STREAM
)

scanner_socket.settimeout(0.5)

result = scanner_socket.connect_ex(
    (TARGET, PORT)
)

if result == 0:
    print("Port", PORT, "is OPEN")
else:
    print("Port", PORT, "is not open")

scanner_socket.close()

## Test Against a Port You Know

A useful scanner test needs at least one service that you know is listening.

You can reuse your **Lab 1 server**.

1. Start your Lab 1 server on `127.0.0.1`.
2. Note its port number, for example `12345`.
3. Change `PORT` in `scanner.py` to that port.
4. Run the scanner.
5. Stop the Lab 1 server.
6. Run the scanner again.

### Observe

When the server is running:

```text
Port 12345 is OPEN
```

When the server is stopped, the connection should no longer succeed.

This gives you a controlled experiment: you know exactly when the port should be open.

# Part 4 — Turn ONE Port Test into a Function

Your one-port code works, but repeating all of it for every port would make the program difficult to read.

Create a function:

```python
def scan_port(target, port):
```

The function should:

- create a new socket,
- set a timeout,
- attempt the connection,
- close the socket,
- return `True` if the connection succeeds,
- return `False` otherwise.

### Why create a new socket for each port?

A socket represents one communication endpoint. Once you use it for a connection attempt, creating a fresh socket for the next port keeps each test independent.

In [ ]:
import socket

def scan_port(target, port):

    sock = socket.socket(
        socket.AF_INET,
        socket.SOCK_STREAM
    )

    sock.settimeout(0.5)

    result = sock.connect_ex(
        (target, port)
    )

    sock.close()

    if result == 0:
        return True
    else:
        return False


print(scan_port("127.0.0.1", 80))

# Part 5 — Scan a Small Range of Ports

Now combine your function with a loop.

Start small.

For example:

```text
20–30
```

or choose a small range that includes the port used by your Lab 1 server.

Do **not** start by scanning all 65,535 TCP ports.

In [ ]:
target = "127.0.0.1"

start_port = 20
end_port = 30

for port in range(start_port, end_port + 1):

    if scan_port(target, port):
        print("Port", port, "is OPEN")

## Your Task

Modify the program so that it also keeps track of discovered open ports.

For example:

```python
open_ports = []
```

When an open port is found, add it to the list.

At the end, display either:

```text
Open ports: [22, 80, 443]
```

or:

```text
No open ports found in the selected range.
```

Do not copy a complete solution. Use the loop and list concepts you already know.

# Part 6 — Identify Likely Services

Port numbers are often associated with common services.

Examples:

```text
22   → SSH
53   → DNS
80   → HTTP
443  → HTTPS
```

Python can look up a common service name with:

```python
socket.getservbyport(port, "tcp")
```

However, not every port has a known mapping, so you need `try / except`.

In [ ]:
import socket

port = 80

try:
    service = socket.getservbyport(
        port,
        "tcp"
    )
except OSError:
    service = "unknown"

print("Port:", port)
print("Likely service:", service)

## Important Security Interpretation

A port number gives us a **clue**, not proof of the application actually running there.

For example:

```text
Port 80 → commonly HTTP
```

does not guarantee that the program listening on port 80 is a web server.

This is one limitation of our simple scanner. Professional scanners can perform additional **service detection**.

# Part 7 — Accept User Input

Now make your scanner interactive.

Ask the user for:

```text
Target host:
Start port:
End port:
```

Example:

```text
Target host: localhost
Start port: 20
End port: 100
```

Use:

- `input()`
- `int()`
- `try / except`
- `socket.gethostbyname()`

Your program should resolve the target before beginning the scan.

In [ ]:
import socket

target = input("Enter target host: ")

try:
    target_ip = socket.gethostbyname(target)
    print("Scanning:", target_ip)

except socket.gaierror:
    print("Invalid hostname or IP address")

# Part 8 — Validate the Port Range

Your scanner should not blindly accept every input.

Check that:

```text
1 <= start_port <= 65535
1 <= end_port <= 65535
start_port <= end_port
```

For this lab, also limit the number of ports scanned in one run.

For example, you may require:

```text
end_port - start_port <= 1000
```

This keeps the exercise controlled and prevents accidental large scans.

If the input is invalid, print a clear message and do not start the scan.

# Part 9 — Build the Complete Scanner

Now combine your work into `scanner.py`.

Your completed program should:

1. ask for a target host,
2. resolve the hostname/IP,
3. ask for a start and end port,
4. validate the port range,
5. scan each TCP port using `scan_port()`,
6. print discovered open ports,
7. show a likely service name when available,
8. report when no open ports are found,
9. handle invalid input without crashing,
10. close every socket that it creates.

A possible output format is:

```text
Target: 127.0.0.1
Scanning TCP ports 20–100...

PORT      STATE      SERVICE
22        open       ssh
80        open       http

Scan complete.
2 open port(s) found.
```

Your output does not need to look exactly like this, but it should be clear and readable.

# Part 10 — Robustness Tests

Demonstrate at least **four** of the following:

1. A known open port
2. A known closed/not-open port
3. Invalid hostname
4. Non-numeric port input
5. Port below 1
6. Port above 65535
7. Start port greater than end port
8. No open ports in the selected range
9. A connection attempt that times out

Your program should handle expected errors gracefully rather than terminating with an unhandled traceback.

### Important

Your simple scanner cannot always distinguish **closed** from **filtered** reliably. Do not label every failed connection as `filtered`. Report conservatively, for example as `not open` or `no successful connection`, unless you have enough information to make a stronger conclusion.

# Part 11 — Compare Your Scanner with Nmap

Now that you understand the scanning logic, compare your program with a professional tool.

Install Nmap if it is not already available.

Run Nmap **only against the same authorized target and small port range** used for your Python scanner.

For example, conceptually:

```text
nmap -p <start>-<end> <authorized-target>
```

Record the results.

### Compare

Answer:

1. Did Nmap identify the same open ports as your Python scanner?
2. What additional information did Nmap provide?
3. Did Nmap distinguish states that your scanner could not?
4. Why is your Python scanner useful for learning even though Nmap is more capable?

# Part 12 — Rebuild Your Scanner Using `python-nmap`

So far, you have built a TCP Connect port scanner using Python's `socket` module. This helped you understand what happens underneath a basic port scan.

Now you will create a **second version** of your scanner using the `python-nmap` library.

`python-nmap` allows a Python program to interact with Nmap and process Nmap scan results programmatically.

## Step 1 — Install `python-nmap`

Make sure **Nmap itself** is installed on your system. Then install the Python library:

```bash
pip install python-nmap
```

> `python-nmap` is a Python interface to Nmap. Installing the Python package does not replace the need to have Nmap installed.

## Step 2 — Import the Library

```python
import nmap
```

## Step 3 — Create a `PortScanner` Object

```python
scanner = nmap.PortScanner()
```

The `PortScanner` object allows your Python program to run Nmap scans and access the results.

## Step 4 — Perform a Small Authorized Scan

Start with `localhost` or another system you are explicitly authorized to scan.

```python
target = "127.0.0.1"

scanner.scan(
    target,
    "20-100"
)
```

Keep the scan range small and controlled, just as you did with your raw socket scanner.

## Step 5 — Explore the Results

Before building the complete program, inspect what Nmap returned.

Try:

```python
print(scanner.all_hosts())
```

Then investigate the protocols discovered for the target:

```python
print(scanner[target].all_protocols())
```

If TCP results are available, inspect them:

```python
print(scanner[target]["tcp"])
```

### Think About It

How is this information different from what your raw socket scanner returned?

## Step 6 — Build `nmap_scanner.py`

Create a second program named:

```text
nmap_scanner.py
```

Your program should:

1. Ask the user for a target.
2. Ask for a start and end port.
3. Validate the input.
4. Perform the scan using `python-nmap`.
5. Display discovered ports.
6. Display the state of each discovered port.
7. Display detected service information when available.
8. Handle errors gracefully.
9. Present the results in a clear format.

For example:

```text
Target: 127.0.0.1

PORT      STATE      SERVICE
22        open       ssh
80        open       http
443       open       https

Scan complete.
```

Do not simply print the entire Nmap result dictionary. Extract the useful information and present it clearly.

## Compare Your Two Implementations

You now have two versions:

```text
scanner.py
     ↓
Python socket
     ↓
TCP connection attempts

nmap_scanner.py
     ↓
python-nmap
     ↓
Nmap
     ↓
Richer scan results
```

Answer the following questions:

1. Which version required more code?
2. Which version provided more information?
3. What information did Nmap provide that your raw socket scanner did not?
4. What work is `python-nmap` hiding from the programmer?
5. Why was it useful to build the raw socket version **before** using `python-nmap`?
6. If you were developing a real security application, which approach would you choose and why?

> **Required:** Submit both `scanner.py` and `nmap_scanner.py`.

# Part 13 — Security Reflection

Answer briefly:

1. What information can an attacker learn from port scanning?
2. Why can an open port increase a system's attack surface?
3. Does an open port automatically mean that a vulnerability exists? Explain.
4. Why should unnecessary network services be disabled or restricted?
5. What is the difference between an open port and a closed port?
6. Why might a firewall make scan results difficult to interpret?
7. Why can our simple TCP Connect scanner not reliably identify every filtered port?
8. How could defenders detect or limit scanning activity?
9. Why should defenders scan their own systems?
10. Why is authorization important before performing network scanning?

# Part 14 — Security Analysis

Write a **300–400 word security analysis** based on your own scan results.

Your analysis must reference:

- the target you were authorized to scan,
- the port range you tested,
- discovered open ports,
- likely services associated with those ports,
- potential risks created by unnecessary exposed services,
- differences you observed between your Python scanner and Nmap,
- defensive measures that could reduce unnecessary exposure.

Do not simply define port scanning. Analyze what your **actual results** mean from a defender's perspective.

# Deliverables

Submit:

- `scanner.py` — your raw Python socket scanner
- `nmap_scanner.py` — your second scanner implemented with `python-nmap`
- a short recording demonstrating:
  - a successful scan of an authorized target,
  - discovery of at least one known open port if available,
  - your selected port range,
  - at least four robustness tests,
  - demonstration of your `python-nmap` version and comparison with the raw socket version,
- answers to the Security Reflection questions,
- a 300–400 word Security Analysis.

Submit **one PDF** containing the security analysis, reflection answers, and cloud links to the required technical artifacts.

### Filename

```text
Lab2-FirstName-LastName-StudentNumber.pdf
```

### Before You Submit

Confirm that:

- your scanner uses raw Python TCP sockets,
- every socket is closed after each connection attempt,
- invalid input is handled,
- the scan range is controlled,
- you scanned only an authorized target,
- your report discusses your own results rather than only general definitions.